# Pygformula pipeline with features identiying L

In [1]:
import numpy as np
import pygformula
from pygformula import ParametricGformula
from pygformula.interventions import static

In [2]:
import pandas as pd

In [3]:
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
#%matplotlib ipympl

# Load MIMIC data

In [5]:
mimicdata = pd.read_parquet('../Dataset/mimic-iv/from_pipeline/ltm_grided_clipped_A_D_Z_Y_SelfPipeline.parquet')

In [6]:
mimicdata.shape

(599188, 42)

In [7]:
mimicdata.columns

Index(['stay_id', 'grid_end', 'vent_mode__hours_since_last__last_12h',
       'temperature__mean__last_12h', 'heart_rate__mean__last_12h',
       'arterial_blood_pressure_mean__mean__last_12h',
       'fluid_out_urine__mean__last_12h', 'pco2_arterial__mean__last_12h',
       'respiratory_rate_measured__mean__last_12h',
       'o2_saturation__mean__last_12h', 'po2_arterial__mean__last_12h',
       'bicarbonate_arterial__last__last_12h',
       'activated_partial_thromboplastin_time__last__last_12h',
       'hemoglobin__last__last_12h', 'creatinine__last__last_12h',
       'ureum__last__last_12h', 'lactate__last__last_12h',
       'glasgow_coma_scale_total__last__last_12h', 'o2_flow__last__last_12h',
       'vent_mode__last__last_12h', 'subject_id', 'raw_age', 'sex',
       'raw_height', 'raw_weight', 'unit_type', 'origin', 'los', 'intime',
       'outtime', 'death_time_from_intime', 'icu_mortality_flag',
       'death_abs_time', 'mortality_after_discharge', 'icu_mortality', 't',
       

In [8]:
mimicdata.rename(columns={"stay_id": "admission_id", "t": "t0", "raw_age":"age", "raw_height":"height", "raw_weight":"weight"}, inplace=True)

In [9]:
mimicdata.vent_mode__last__last_12h = mimicdata.vent_mode__last__last_12h.astype("category")

In [10]:
mimicdata.vent_mode__last__last_12h.value_counts()

vent_mode__last__last_12h
unknown                522887
invasive_controlled     56730
invasive_assisted       19525
cancelled                  46
Name: count, dtype: int64

In [11]:
mimicdata.o2_flow__last__last_12h.describe()

count    45301.000000
mean        46.678171
std         13.793587
min          7.342648
25%         40.000000
50%         40.000000
75%         50.000000
max         87.342648
Name: o2_flow__last__last_12h, dtype: float64

# Declare baseline variables and meta variables

In [12]:
#baseline = ['age',
#            'gender',
#            'ethnicity',
#            'insurance'
#            ]
baseline = ["age", 
            "sex", 
            #"height", 
            #"weight", 
            #"unit_type", 
            "origin"]

In [13]:
# Pacmed - 92% have been discharged from the ICU. The rest died inside the ICU which tallied with 7.8% that have D=1
mimicdata.groupby('admission_id').tail(1)['A'].mean()

np.float64(0.9261027952268641)

# Deaths

In [14]:
obs_data = mimicdata.sort_values(["admission_id", "t0"]).copy()

# Initialize Y as NaN everywhere
obs_data["Y"] = 0

# Identify last row per stay
last_idx = obs_data.groupby(id, sort=False).tail(1).index

# Extract needed columns on last rows
A_last = obs_data.loc[last_idx, "A"]
I_last = obs_data.loc[last_idx, "D"]
Z_last = obs_data.loc[last_idx, "Z"]

# Compute Y_hat according to your rule (without filling NaNs globally)
Y_last = (
    (I_last == 1) |
    ((A_last == 1) & (Z_last == 1))
).astype(int)

# Assign only to terminal rows
obs_data.loc[last_idx, "Y"] = Y_last
obs_mean_Ey = obs_data.loc[last_idx, "Y"].mean()
obs_mean_Ey

np.float64(0.02909270546139108)

In [15]:
last_rows = (
        obs_data.sort_values(['admission_id', 't0'])
          .groupby('admission_id')
          .tail(1)
    )
    
D_mean = last_rows['D'].fillna(0).mean()
Z_mean = last_rows['Z'].fillna(0).mean()
Y_mean = last_rows['Y'].mean()

results = []
label = "MIMIC"
results.append({
    "Strategy": label,
    "Mean D (stay-level)": D_mean,
    "Mean Z (stay-level)": Z_mean,
    "Mean Y (stay-level)": Y_mean
})
summary_table = pd.DataFrame(results)
summary_table

,Strategy,Mean D (stay-level),Mean Z (stay-level),Mean Y (stay-level)
0,MIMIC,0.073745,0.12979,0.203535


In [16]:
mimicdata[['Y', 'D']].drop_duplicates(ignore_index=True)

,Y,D
0,NaN,0.0
1,0.0,NaN
2,1.0,NaN
3,1.0,1.0
4,0.0,0.0


In [17]:
mimicdata.groupby('admission_id').tail(1)[['Y', 'D']].drop_duplicates(ignore_index=True)

,Y,D
0,0.0,NaN
1,1.0,NaN
2,1.0,1.0
3,0.0,0.0


In [18]:
picked_L = [
    'vent_mode__last__last_12h',
    'vent_mode__hours_since_last__last_12h',
    
    'bicarbonate_arterial__last__last_12h',
    'activated_partial_thromboplastin_time__last__last_12h',
    'temperature__mean__last_12h',

    'hemoglobin__last__last_12h',
    'heart_rate__mean__last_12h',
    'arterial_blood_pressure_mean__mean__last_12h',

    'creatinine__last__last_12h',
    'ureum__last__last_12h',
    'fluid_out_urine__mean__last_12h',
    'lactate__last__last_12h',
    
    'glasgow_coma_scale_total__last__last_12h',
    
    'pco2_arterial__mean__last_12h',
    'respiratory_rate_measured__mean__last_12h',
    'o2_saturation__mean__last_12h',
    'o2_flow__last__last_12h',  
    'po2_arterial__mean__last_12h',
]

In [19]:
mimicdata.admission_id.nunique()

85646

## Handle missing values

In [20]:
# Columns to forward fill
feature_columns = list(picked_L)

In [21]:
mimicdata[feature_columns].isna().sum()

vent_mode__last__last_12h                                     0
vent_mode__hours_since_last__last_12h                    453814
bicarbonate_arterial__last__last_12h                     582973
activated_partial_thromboplastin_time__last__last_12h    588525
temperature__mean__last_12h                               48046
hemoglobin__last__last_12h                               582830
heart_rate__mean__last_12h                                 2199
arterial_blood_pressure_mean__mean__last_12h              16526
creatinine__last__last_12h                               582967
ureum__last__last_12h                                    582980
fluid_out_urine__mean__last_12h                          202848
lactate__last__last_12h                                  589939
glasgow_coma_scale_total__last__last_12h                 516053
pco2_arterial__mean__last_12h                            370415
respiratory_rate_measured__mean__last_12h                  5107
o2_saturation__mean__last_12h           

In [22]:
mimicdata.loc[mimicdata.admission_id=="mimic4-39996123"][["grid_end", "vent_mode__last__last_12h", "vent_mode__hours_since_last__last_12h"]]

,grid_end,vent_mode__last__last_12h,vent_mode__hours_since_last__last_12h
598985,0 days 12:00:00,invasive_controlled,1.783333
598986,1 days 00:00:00,invasive_assisted,5.366667
598987,1 days 12:00:00,unknown,NaN
598988,2 days 00:00:00,unknown,NaN
598989,2 days 12:00:00,unknown,NaN
598990,3 days 00:00:00,unknown,NaN


In [23]:
# # Apply forward fill per group, only on selected columns; Then backward fill to handle top-level NaNs per group
mimicdata[feature_columns] = mimicdata.groupby('admission_id')[feature_columns].ffill().bfill()

In [24]:
mimicdata.loc[mimicdata.admission_id=="mimic4-39996123"][["grid_end", "vent_mode__last__last_12h", "vent_mode__hours_since_last__last_12h"]]

,grid_end,vent_mode__last__last_12h,vent_mode__hours_since_last__last_12h
598985,0 days 12:00:00,invasive_controlled,1.783333
598986,1 days 00:00:00,invasive_assisted,5.366667
598987,1 days 12:00:00,unknown,5.366667
598988,2 days 00:00:00,unknown,5.366667
598989,2 days 12:00:00,unknown,5.366667
598990,3 days 00:00:00,unknown,5.366667


In [25]:
mimicdata[feature_columns].isna().sum()

vent_mode__last__last_12h                                 0
vent_mode__hours_since_last__last_12h                    25
bicarbonate_arterial__last__last_12h                     47
activated_partial_thromboplastin_time__last__last_12h    47
temperature__mean__last_12h                               0
hemoglobin__last__last_12h                               47
heart_rate__mean__last_12h                                0
arterial_blood_pressure_mean__mean__last_12h              0
creatinine__last__last_12h                               47
ureum__last__last_12h                                    47
fluid_out_urine__mean__last_12h                          16
lactate__last__last_12h                                  87
glasgow_coma_scale_total__last__last_12h                  0
pco2_arterial__mean__last_12h                            25
respiratory_rate_measured__mean__last_12h                 0
o2_saturation__mean__last_12h                             0
o2_flow__last__last_12h                 

In [26]:
# Find rows where any of these columns are NaN
mask_missing = mimicdata[feature_columns].isna().any(axis=1)

# Get corresponding admission IDs
missing_admissions = mimicdata.loc[mask_missing, "admission_id"].unique()

print("Admissions with any missing feature:", missing_admissions)
print("Total:", len(missing_admissions))


Admissions with any missing feature: ['mimic4-39998138' 'mimic4-39998622' 'mimic4-39999168' 'mimic4-39999172'
 'mimic4-39999230' 'mimic4-39999286' 'mimic4-39999301' 'mimic4-39999384'
 'mimic4-39999552' 'mimic4-39999562' 'mimic4-39999810' 'mimic4-39999858']
Total: 12


In [27]:
mimicdata.loc[mimicdata['admission_id']=='mimic4-39941251'].isna().sum()

admission_id                                             0
grid_end                                                 0
vent_mode__hours_since_last__last_12h                    0
temperature__mean__last_12h                              0
heart_rate__mean__last_12h                               0
arterial_blood_pressure_mean__mean__last_12h             0
fluid_out_urine__mean__last_12h                          0
pco2_arterial__mean__last_12h                            0
respiratory_rate_measured__mean__last_12h                0
o2_saturation__mean__last_12h                            0
po2_arterial__mean__last_12h                             0
bicarbonate_arterial__last__last_12h                     0
activated_partial_thromboplastin_time__last__last_12h    0
hemoglobin__last__last_12h                               0
creatinine__last__last_12h                               0
ureum__last__last_12h                                    0
lactate__last__last_12h                                 

In [28]:
mimicdata = mimicdata[~mimicdata['admission_id'].isin(missing_admissions)]

# Check missingness including baseline variables

In [29]:
#check missingness including baseline variables
all = feature_columns + baseline
mimicdata[all].isna().sum()

vent_mode__last__last_12h                                    0
vent_mode__hours_since_last__last_12h                        0
bicarbonate_arterial__last__last_12h                         0
activated_partial_thromboplastin_time__last__last_12h        0
temperature__mean__last_12h                                  0
hemoglobin__last__last_12h                                   0
heart_rate__mean__last_12h                                   0
arterial_blood_pressure_mean__mean__last_12h                 0
creatinine__last__last_12h                                   0
ureum__last__last_12h                                        0
fluid_out_urine__mean__last_12h                              0
lactate__last__last_12h                                      0
glasgow_coma_scale_total__last__last_12h                     0
pco2_arterial__mean__last_12h                                0
respiratory_rate_measured__mean__last_12h                    0
o2_saturation__mean__last_12h                          

In [30]:
# Find rows where any of these columns are NaN
mask_missing = mimicdata[all].isna().any(axis=1)

# Get corresponding admission IDs
missing_admissions = mimicdata.loc[mask_missing, "admission_id"].unique()

print("Admissions with any missing feature:", missing_admissions)
print("Total:", len(missing_admissions))

Admissions with any missing feature: ['mimic4-30001555' 'mimic4-30004232' 'mimic4-30004320' ...
 'mimic4-39986935' 'mimic4-39987185' 'mimic4-39997370']
Total: 3154


In [31]:
mimicdata.loc[mimicdata.admission_id=="mimic4-30001555"].sex

35   NaN
36   NaN
Name: sex, dtype: float64

In [32]:
mimicdata = mimicdata[~mimicdata['admission_id'].isin(missing_admissions)]

In [33]:
mimicdata[all].isna().sum()

vent_mode__last__last_12h                                0
vent_mode__hours_since_last__last_12h                    0
bicarbonate_arterial__last__last_12h                     0
activated_partial_thromboplastin_time__last__last_12h    0
temperature__mean__last_12h                              0
hemoglobin__last__last_12h                               0
heart_rate__mean__last_12h                               0
arterial_blood_pressure_mean__mean__last_12h             0
creatinine__last__last_12h                               0
ureum__last__last_12h                                    0
fluid_out_urine__mean__last_12h                          0
lactate__last__last_12h                                  0
glasgow_coma_scale_total__last__last_12h                 0
pco2_arterial__mean__last_12h                            0
respiratory_rate_measured__mean__last_12h                0
o2_saturation__mean__last_12h                            0
o2_flow__last__last_12h                                 

In [34]:
mimicdata['t0'] = mimicdata['t0'].astype(int)

# Normalizing numeric columns

In [35]:
los_per_admission = mimicdata.groupby("admission_id")["t0"].max().reset_index(name="LOS")
los_per_admission["LOS"] = los_per_admission["LOS"] + 1

In [36]:
los_per_admission

,admission_id,LOS
0,mimic4-30000153,2
1,mimic4-30000213,2
2,mimic4-30000484,4
3,mimic4-30000646,8
4,mimic4-30000831,4
...,...,...
82475,mimic4-39997159,10
82476,mimic4-39997710,27
82477,mimic4-39997753,3
82478,mimic4-39997955,9


In [37]:
unique_los = los_per_admission["LOS"].unique()
unique_los.sort()
print(unique_los)


[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  95  96  97  98  99 100 101 102 103 104 105 106 107 108 109
 110 111 113 114 115 116 117 118 119 121 123 125 126 127 129 132 134 135
 137 139 140 143 144 147 148 149 150 151 154 155 157 162 163 165 173 175
 181]


In [38]:
# LOS per admission (assuming t0 starts at 0)
los_per_admission = (
    mimicdata.groupby("admission_id")["t0"].max().reset_index(name="LOS")
)
los_per_admission["LOS"] = los_per_admission["LOS"] + 1

# Distribution of LOS
los_distribution = los_per_admission["LOS"].value_counts().sort_index()

print(los_distribution)

LOS
1      18638
2      12314
3      11648
4       6279
5       6299
       ...  
163        1
165        2
173        1
175        1
181       12
Name: count, Length: 145, dtype: int64


In [39]:
los_distribution.sort_values()

LOS
139        1
114        1
129        1
137        1
135        1
       ...  
4       6279
5       6299
3      11648
2      12314
1      18638
Name: count, Length: 145, dtype: int64

In [40]:
#los_distribution.sort_values().to_csv("los_dist.csv", index=True)

In [41]:
#mimicdata = mimicdata[mimicdata["los"] <= pd.to_timedelta(100, unit="D")].copy()

In [42]:
mimicdata.admission_id.nunique()

82480

In [43]:
# Cut O2 saturation to 100%
mimicdata["o2_saturation__mean__last_12h"] = (
    mimicdata["o2_saturation__mean__last_12h"].clip(upper=100)
)

# Crafting discharge policies

In [44]:
# variable list
covnames = [
    'vent_mode__last__last_12h',
    'vent_mode__hours_since_last__last_12h',
    
    'pco2_arterial__mean__last_12h',
    'po2_arterial__mean__last_12h',
    'o2_flow__last__last_12h',
    'o2_saturation__mean__last_12h',
    'respiratory_rate_measured__mean__last_12h',
    
    'glasgow_coma_scale_total__last__last_12h',

    'lactate__last__last_12h',
    'fluid_out_urine__mean__last_12h',
    'ureum__last__last_12h',
    'creatinine__last__last_12h',

    'arterial_blood_pressure_mean__mean__last_12h',
    'heart_rate__mean__last_12h',
    'hemoglobin__last__last_12h',
    
    'temperature__mean__last_12h',
    'activated_partial_thromboplastin_time__last__last_12h',
    'bicarbonate_arterial__last__last_12h',
    
    'A'
]

cat = ['vent_mode__last__last_12h']

def summarize_variable(series, ndigits=2, is_categorical=False):
    
    # Categorical variable
    if is_categorical:
        value_counts = series.value_counts(dropna=False)
        return {
            "counts": value_counts.to_dict(),
        }
    
    # Numeric variable
    if not series.notna().any():
        return {"interval": None, "mean": None, "median": None}

    min_v = float(series.min(skipna=True))
    max_v = float(series.max(skipna=True))
    mean_v = float(series.mean(skipna=True))
    med_v = float(series.median(skipna=True))

    return {
        "interval": f"{round(min_v, ndigits)} – {round(max_v, ndigits)}",
        "mean": round(mean_v, ndigits),
        "median": round(med_v, ndigits)
    }


# Filter to discharge events
df_discharge = mimicdata[mimicdata["A"] == 1]

vars_to_summarize = [v for v in covnames if v != "A"]

ranges_dict = {
    var: summarize_variable(
            df_discharge[var],
            ndigits=2,
            is_categorical=(var in cat)
         )
    for var in vars_to_summarize
}

ranges_dict

{'vent_mode__last__last_12h': {'counts': {'unknown': 74171,
   'invasive_assisted': 1340,
   'invasive_controlled': 867,
   'cancelled': 2}},
 'vent_mode__hours_since_last__last_12h': {'interval': '0.0 – 12.0',
  'mean': 4.2,
  'median': 3.27},
 'pco2_arterial__mean__last_12h': {'interval': '0.0 – 91.12',
  'mean': 42.12,
  'median': 41.0},
 'po2_arterial__mean__last_12h': {'interval': '12.33 – 336.02',
  'mean': 120.75,
  'median': 98.5},
 'o2_flow__last__last_12h': {'interval': '7.34 – 87.34',
  'mean': 48.13,
  'median': 40.0},
 'o2_saturation__mean__last_12h': {'interval': '83.98 – 100.0',
  'mean': 96.35,
  'median': 96.46},
 'respiratory_rate_measured__mean__last_12h': {'interval': '0.0 – 43.92',
  'mean': 19.0,
  'median': 18.5},
 'glasgow_coma_scale_total__last__last_12h': {'interval': '3.0 – 15.0',
  'mean': 12.62,
  'median': 15.0},
 'lactate__last__last_12h': {'interval': '0.3 – 8.12',
  'mean': 2.29,
  'median': 1.6},
 'fluid_out_urine__mean__last_12h': {'interval': '0.0 – 

In [45]:
mimicdata.admission_id.nunique()

82480

In [46]:
# Sub sample of admissions
rng = np.random.default_rng(34)
unique_ids = mimicdata['admission_id'].dropna().unique()
sampled_ids = rng.choice(unique_ids, size=10000, replace=False)
mimicdata_sub = mimicdata[mimicdata['admission_id'].isin(sampled_ids)].copy()
mimicdata_sub.reset_index(drop=True, inplace=True)

In [47]:
mimicdata.to_parquet("gdata.parquet")

In [48]:
(
    mimicdata.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['Y']
             .mean()
)

np.float64(0.20081231813773037)

In [49]:
mimicdata.groupby('admission_id')['Y'].max().mean()

np.float64(0.20081231813773037)

In [50]:
stay = mimicdata.groupby("admission_id").agg(D=("D","max"), A=("A","max"), Z=("Z","max"))

Z1 = (stay["Z"] == 1).astype(int)          # NaN -> 0 automatically
Y  = ((stay["D"] == 1) | ((stay["A"] == 1) & (stay["Z"] == 1))).astype(int)

stay["Y"] = Y

In [51]:
stay.Y.mean()

np.float64(0.20081231813773037)

In [52]:
mimicdata.admission_id.nunique()

82480

# Applying g-formula

In [53]:
time_name = 't0'
id_ = 'admission_id'

In [54]:
basecovs = baseline
'''['age',
            'gender',
            'ethnicity',
            'insurance'
            ]'''

"['age',\n            'gender',\n            'ethnicity',\n            'insurance'\n            ]"

In [55]:
covnames = [
    'vent_mode__last__last_12h',
    'vent_mode__hours_since_last__last_12h',
    
    'pco2_arterial__mean__last_12h',
    'po2_arterial__mean__last_12h',
    'o2_flow__last__last_12h',
    'o2_saturation__mean__last_12h',
    'respiratory_rate_measured__mean__last_12h',
    
    'glasgow_coma_scale_total__last__last_12h',

    'lactate__last__last_12h',
    'fluid_out_urine__mean__last_12h',
    'ureum__last__last_12h',
    'creatinine__last__last_12h',

    'arterial_blood_pressure_mean__mean__last_12h',
    'heart_rate__mean__last_12h',
    'hemoglobin__last__last_12h',
    
    'temperature__mean__last_12h',
    'activated_partial_thromboplastin_time__last__last_12h',
    'bicarbonate_arterial__last__last_12h',
    
    'A'
]

In [56]:
covtypes = [
 'categorical',          #'vent_mode__last__last_12h'
 'zero-inflated normal', #'zero-inflated normal', #'vent_mode__hours_since_last__last_12h'

 'truncated normal',       #'pco2_arterial__mean__last_12h'
 'bounded normal',       #'po2_arterial__mean__last_12h'
 'bounded normal',       #'o2_flow__last__last_12h',
 'bounded normal',       #'o2_saturation__mean__last_12h',
 'bounded normal',       #'respiratory_rate_measured__mean__last_12h',

 'bounded normal',       #'glasgow_coma_scale_total__last__last_12h',
 'truncated normal',     #'lactate__last__last_12h',

 'zero-inflated normal', #'fluid_out_urine__mean__last_12h',
 'truncated normal',     #'ureum__last__last_12h',
 'truncated normal',     #'creatinine__last__last_12h',

 'bounded normal',       #'arterial_blood_pressure_mean__mean__last_12h',
 'bounded normal',       #'heart_rate__mean__last_12h',
 'bounded normal',       #'hemoglobin__last__last_12h'
    
 'normal',               #'temperature__mean__last_12h',
 'truncated normal',     #'activated_partial_thromboplastin_time__last__last_12h',
 'bounded normal',       #'bicarbonate_arterial__last__last_12h'

 'binary'              #'A'
]

In [57]:
trunc_params = [
    # vent_mode__last__last_12h
    "NA",

    # vent_mode__hours_since_last__last_12h
    "NA",

    # pco2_arterial__mean__last_12h
    [0, "left"],

    # po2_arterial__mean__last_12h
    [0, "left"],

    # o2_flow__last__last_12h
    "NA",

    # o2_saturation__mean__last_12h
    "NA",

    # respiratory_rate_measured__mean__last_12h
    "NA",

    # glasgow_coma_scale_total__last__last_12h
    "NA",

    # lactate__last__last_12h
    [0, "left"],

    # fluid_out_urine__mean__last_12h
    [0, "left"],

    # ureum__last__last_12h
    [0, "left"],

    # creatinine__last__last_12h
    [0, "left"],

    # arterial_blood_pressure_mean__mean__last_12h
    "NA",

    # heart_rate__mean__last_12h
    "NA",

    # hemoglobin__last__last_12h
    "NA",

    # temperature__mean__last_12h
    "NA",

    # activated_partial_thromboplastin_time__last__last_12h
    [0, "left"],

    # bicarbonate_arterial__last__last_12h
    "NA",

    #A
    "NA"
]


In [58]:
'''covmodels = [ 
 #'vent_mode__last__last_12h ~ glasgow_coma_scale_total__last__last_12h + po2_arterial__mean__last_12h + o2_saturation__mean__last_12h + respiratory_rate_measured__mean__last_12h',    
 
 #'vent_mode__hours_since_last__last_12h ~ lag1_vent_mode__hours_since_last__last_12h + vent_mode__last__last_12h',
 'vent_mode__hours_since_last__last_12h ~ lag1_vent_mode__hours_since_last__last_12h',

 'pco2_arterial__mean__last_12h ~ lag1_pco2_arterial__mean__last_12h + o2_flow__last__last_12h', 
 'po2_arterial__mean__last_12h ~ lag1_po2_arterial__mean__last_12h + o2_flow__last__last_12h + pco2_arterial__mean__last_12h',
 
 #'o2_flow__last__last_12h ~ lag1_o2_flow__last__last_12h + vent_mode__last__last_12h + respiratory_rate_measured__mean__last_12h',
 'o2_flow__last__last_12h ~ lag1_o2_flow__last__last_12h + respiratory_rate_measured__mean__last_12h',
 
 'o2_saturation__mean__last_12h ~ lag1_o2_saturation__mean__last_12h + po2_arterial__mean__last_12h + respiratory_rate_measured__mean__last_12h',
 
 'respiratory_rate_measured__mean__last_12h ~ lag1_respiratory_rate_measured__mean__last_12h + glasgow_coma_scale_total__last__last_12h',
 #'respiratory_rate_measured__mean__last_12h ~ lag1_respiratory_rate_measured__mean__last_12h',

 'glasgow_coma_scale_total__last__last_12h ~ lag1_glasgow_coma_scale_total__last__last_12h + arterial_blood_pressure_mean__mean__last_12h + lactate__last__last_12h',
 
 'lactate__last__last_12h ~ lag1_lactate__last__last_12h + arterial_blood_pressure_mean__mean__last_12h + fluid_out_urine__mean__last_12h',
 #'lactate__last__last_12h ~ lag1_lactate__last__last_12h + arterial_blood_pressure_mean__mean__last_12h',

 'fluid_out_urine__mean__last_12h ~ lag1_fluid_out_urine__mean__last_12h + arterial_blood_pressure_mean__mean__last_12h + ureum__last__last_12h',
 'ureum__last__last_12h ~ lag1_ureum__last__last_12h + creatinine__last__last_12h',
 'creatinine__last__last_12h ~ lag1_creatinine__last__last_12h + arterial_blood_pressure_mean__mean__last_12h',

 'arterial_blood_pressure_mean__mean__last_12h ~ lag1_arterial_blood_pressure_mean__mean__last_12h + heart_rate__mean__last_12h + bicarbonate_arterial__last__last_12h',
 'heart_rate__mean__last_12h ~ lag1_heart_rate__mean__last_12h + temperature__mean__last_12h + hemoglobin__last__last_12h',
 'hemoglobin__last__last_12h ~ lag1_hemoglobin__last__last_12h + activated_partial_thromboplastin_time__last__last_12h',
    
 'temperature__mean__last_12h ~ lag1_temperature__mean__last_12h',
 'activated_partial_thromboplastin_time__last__last_12h ~ lag1_activated_partial_thromboplastin_time__last__last_12h',
 'bicarbonate_arterial__last__last_12h ~ lag1_bicarbonate_arterial__last__last_12h',

 
 #'A ~ vent_mode__last__last_12h + vent_mode__hours_since_last__last_12h + po2_arterial__mean__last_12h + o2_flow__last__last_12h + o2_saturation__mean__last_12h + respiratory_rate_measured__mean__last_12h + glasgow_coma_scale_total__last__last_12h + lactate__last__last_12h + fluid_out_urine__mean__last_12h + ureum__last__last_12h + creatinine__last__last_12h + arterial_blood_pressure_mean__mean__last_12h + heart_rate__mean__last_12h + hemoglobin__last__last_12h + temperature__mean__last_12h + activated_partial_thromboplastin_time__last__last_12h + bicarbonate_arterial__last__last_12h']
 'A ~ vent_mode__hours_since_last__last_12h + po2_arterial__mean__last_12h + o2_flow__last__last_12h + o2_saturation__mean__last_12h + respiratory_rate_measured__mean__last_12h + glasgow_coma_scale_total__last__last_12h + lactate__last__last_12h + fluid_out_urine__mean__last_12h + ureum__last__last_12h + creatinine__last__last_12h + arterial_blood_pressure_mean__mean__last_12h + heart_rate__mean__last_12h + hemoglobin__last__last_12h + temperature__mean__last_12h + activated_partial_thromboplastin_time__last__last_12h + bicarbonate_arterial__last__last_12h']'''


"covmodels = [ \n #'vent_mode__last__last_12h ~ glasgow_coma_scale_total__last__last_12h + po2_arterial__mean__last_12h + o2_saturation__mean__last_12h + respiratory_rate_measured__mean__last_12h',    \n\n #'vent_mode__hours_since_last__last_12h ~ lag1_vent_mode__hours_since_last__last_12h + vent_mode__last__last_12h',\n 'vent_mode__hours_since_last__last_12h ~ lag1_vent_mode__hours_since_last__last_12h',\n\n 'pco2_arterial__mean__last_12h ~ lag1_pco2_arterial__mean__last_12h + o2_flow__last__last_12h', \n 'po2_arterial__mean__last_12h ~ lag1_po2_arterial__mean__last_12h + o2_flow__last__last_12h + pco2_arterial__mean__last_12h',\n\n #'o2_flow__last__last_12h ~ lag1_o2_flow__last__last_12h + vent_mode__last__last_12h + respiratory_rate_measured__mean__last_12h',\n 'o2_flow__last__last_12h ~ lag1_o2_flow__last__last_12h + respiratory_rate_measured__mean__last_12h',\n\n 'o2_saturation__mean__last_12h ~ lag1_o2_saturation__mean__last_12h + po2_arterial__mean__last_12h + respiratory_rate_m

In [59]:
covmodels = [

# -------------------------
# 0) Vent support intensity proxy (hours since last) — treat as support/measurement intensity
# -------------------------

"vent_mode__hours_since_last__last_12h ~ "
"lag1_vent_mode__hours_since_last__last_12h + lag2_vent_mode__hours_since_last__last_12h"
" + lag1_po2_arterial__mean__last_12h + lag1_pco2_arterial__mean__last_12h"
" + lag1_o2_saturation__mean__last_12h + lag1_respiratory_rate_measured__mean__last_12h"
" + lag1_glasgow_coma_scale_total__last__last_12h",

# -------------------------
# 1) Gas exchange (depends on prior support intensity + prior state)
# -------------------------

"pco2_arterial__mean__last_12h ~ "
"lag1_pco2_arterial__mean__last_12h + lag2_pco2_arterial__mean__last_12h"
" + lag1_o2_flow__last__last_12h"
" + lag1_vent_mode__hours_since_last__last_12h"
" + lag1_respiratory_rate_measured__mean__last_12h",

"po2_arterial__mean__last_12h ~ "
"lag1_po2_arterial__mean__last_12h + lag2_po2_arterial__mean__last_12h"
" + lag1_o2_flow__last__last_12h"
" + lag1_vent_mode__hours_since_last__last_12h"
" + lag1_arterial_blood_pressure_mean__mean__last_12h",



# -------------------------
# 2) Support delivery (avoid contemporaneous RR_t)
# -------------------------

"o2_flow__last__last_12h ~ "
"lag1_o2_flow__last__last_12h + lag2_o2_flow__last__last_12h"
" + lag1_po2_arterial__mean__last_12h + lag1_o2_saturation__mean__last_12h"
" + lag1_vent_mode__hours_since_last__last_12h"
" + lag1_respiratory_rate_measured__mean__last_12h",

    
"o2_saturation__mean__last_12h ~ "
"lag1_o2_saturation__mean__last_12h + lag2_o2_saturation__mean__last_12h"
" + lag1_po2_arterial__mean__last_12h"
" + lag1_o2_flow__last__last_12h"
" + lag1_respiratory_rate_measured__mean__last_12h",

# -------------------------
# 3) Vitals / neuro (respond to prior gases + perfusion; no within-bin loops)
# -------------------------

"respiratory_rate_measured__mean__last_12h ~ "
"lag1_respiratory_rate_measured__mean__last_12h + lag2_respiratory_rate_measured__mean__last_12h"
" + lag1_po2_arterial__mean__last_12h + lag1_pco2_arterial__mean__last_12h"
" + lag1_o2_saturation__mean__last_12h"
" + lag1_glasgow_coma_scale_total__last__last_12h",

"glasgow_coma_scale_total__last__last_12h ~ "
"lag1_glasgow_coma_scale_total__last__last_12h + lag2_glasgow_coma_scale_total__last__last_12h"
" + lag1_pco2_arterial__mean__last_12h"
" + lag1_arterial_blood_pressure_mean__mean__last_12h"
" + lag1_lactate__last__last_12h",


# -------------------------
# 4) Metabolic / acid-base
# -------------------------

"lactate__last__last_12h ~ "
"lag1_lactate__last__last_12h + lag2_lactate__last__last_12h"
" + lag1_arterial_blood_pressure_mean__mean__last_12h"
" + lag1_fluid_out_urine__mean__last_12h"
" + lag1_temperature__mean__last_12h",

# -------------------------
# 5) Renal (slow; lag2 important)
# -------------------------

"fluid_out_urine__mean__last_12h ~ "
"lag1_fluid_out_urine__mean__last_12h + lag2_fluid_out_urine__mean__last_12h"
" + lag1_arterial_blood_pressure_mean__mean__last_12h"
" + lag1_creatinine__last__last_12h + lag1_ureum__last__last_12h",

"ureum__last__last_12h ~ "
"lag1_ureum__last__last_12h + lag2_ureum__last__last_12h"
" + lag1_creatinine__last__last_12h"
" + lag1_fluid_out_urine__mean__last_12h",
    
    
"creatinine__last__last_12h ~ "
"lag1_creatinine__last__last_12h + lag2_creatinine__last__last_12h"
" + lag1_fluid_out_urine__mean__last_12h"
" + lag1_arterial_blood_pressure_mean__mean__last_12h",

"arterial_blood_pressure_mean__mean__last_12h ~ "
"lag1_arterial_blood_pressure_mean__mean__last_12h + lag2_arterial_blood_pressure_mean__mean__last_12h"
" + lag1_heart_rate__mean__last_12h"
" + lag1_lactate__last__last_12h"
" + lag1_bicarbonate_arterial__last__last_12h",

"heart_rate__mean__last_12h ~ "
"lag1_heart_rate__mean__last_12h + lag2_heart_rate__mean__last_12h"
" + lag1_temperature__mean__last_12h"
" + lag1_arterial_blood_pressure_mean__mean__last_12h"
" + lag1_lactate__last__last_12h"
" + lag1_hemoglobin__last__last_12h",


# -------------------------
# 6) Hematology / coag (slow; lag2 important)
# -------------------------

"hemoglobin__last__last_12h ~ "
"lag1_hemoglobin__last__last_12h + lag2_hemoglobin__last__last_12h"
" + lag1_activated_partial_thromboplastin_time__last__last_12h",

"temperature__mean__last_12h ~ "
"lag1_temperature__mean__last_12h + lag2_temperature__mean__last_12h"
" + lag1_lactate__last__last_12h",


"activated_partial_thromboplastin_time__last__last_12h ~ "
"lag1_activated_partial_thromboplastin_time__last__last_12h + lag2_activated_partial_thromboplastin_time__last__last_12h",

"bicarbonate_arterial__last__last_12h ~ "
"lag1_bicarbonate_arterial__last__last_12h + lag2_bicarbonate_arterial__last__last_12h"
" + lag1_lactate__last__last_12h"
" + lag1_pco2_arterial__mean__last_12h"
" + lag1_creatinine__last__last_12h",


# -------------------------
# 7) Discharge hazard A (depends on prior stability + organ support; include time-in-ICU)
# -------------------------

"A ~ t0"
" + lag1_vent_mode__hours_since_last__last_12h"
" + lag1_po2_arterial__mean__last_12h + lag1_o2_flow__last__last_12h + lag1_o2_saturation__mean__last_12h"
" + lag1_respiratory_rate_measured__mean__last_12h"
" + lag1_glasgow_coma_scale_total__last__last_12h"
" + lag1_lactate__last__last_12h + lag1_bicarbonate_arterial__last__last_12h"
" + lag1_fluid_out_urine__mean__last_12h + lag1_ureum__last__last_12h + lag1_creatinine__last__last_12h"
" + lag1_arterial_blood_pressure_mean__mean__last_12h + lag1_heart_rate__mean__last_12h"
" + lag1_hemoglobin__last__last_12h + lag1_temperature__mean__last_12h"
" + lag1_activated_partial_thromboplastin_time__last__last_12h"

]

In [60]:
outcome_name = 'Z'
# Outcome model
#zmodel = 'Z ~ vent_mode__last__last_12h + cumavg_vent_mode__hours_since_last__last_12h + cumavg_pco2_arterial__mean__last_12h + cumavg_po2_arterial__mean__last_12h + cumavg_o2_flow__last__last_12h + cumavg_o2_saturation__mean__last_12h + cumavg_respiratory_rate_measured__mean__last_12h + cumavg_glasgow_coma_scale_total__last__last_12h + cumavg_lactate__last__last_12h + cumavg_fluid_out_urine__mean__last_12h + cumavg_ureum__last__last_12h + cumavg_creatinine__last__last_12h + cumavg_arterial_blood_pressure_mean__mean__last_12h + cumavg_heart_rate__mean__last_12h + cumavg_hemoglobin__last__last_12h + cumavg_temperature__mean__last_12h + cumavg_activated_partial_thromboplastin_time__last__last_12h + cumavg_bicarbonate_arterial__last__last_12h'
zmodel = 'Z ~ cumavg_vent_mode__hours_since_last__last_12h + cumavg_pco2_arterial__mean__last_12h + cumavg_po2_arterial__mean__last_12h + cumavg_o2_flow__last__last_12h + cumavg_o2_saturation__mean__last_12h + cumavg_respiratory_rate_measured__mean__last_12h + cumavg_glasgow_coma_scale_total__last__last_12h + cumavg_lactate__last__last_12h + cumavg_fluid_out_urine__mean__last_12h + cumavg_ureum__last__last_12h + cumavg_creatinine__last__last_12h + cumavg_arterial_blood_pressure_mean__mean__last_12h + cumavg_heart_rate__mean__last_12h + cumavg_hemoglobin__last__last_12h + cumavg_temperature__mean__last_12h + cumavg_activated_partial_thromboplastin_time__last__last_12h + cumavg_bicarbonate_arterial__last__last_12h'

'''zmodel = (
"Z ~ "
" + vent_mode__hours_since_last__last_12h"
" + pco2_arterial__mean__last_12h + po2_arterial__mean__last_12h"
" + o2_flow__last__last_12h + o2_saturation__mean__last_12h"
" + respiratory_rate_measured__mean__last_12h"
" + glasgow_coma_scale_total__last__last_12h"
" + lactate__last__last_12h"
" + fluid_out_urine__mean__last_12h + ureum__last__last_12h + creatinine__last__last_12h"
" + arterial_blood_pressure_mean__mean__last_12h + heart_rate__mean__last_12h"
" + hemoglobin__last__last_12h"
" + temperature__mean__last_12h"
" + activated_partial_thromboplastin_time__last__last_12h"
" + bicarbonate_arterial__last__last_12h"
)'''


outcome_type = 'binary_eof'

In [61]:
I_name = 'D'
#I_model = 'D ~ vent_mode__last__last_12h + vent_mode__hours_since_last__last_12h + pco2_arterial__mean__last_12h + po2_arterial__mean__last_12h + o2_flow__last__last_12h + o2_saturation__mean__last_12h + respiratory_rate_measured__mean__last_12h + glasgow_coma_scale_total__last__last_12h + lactate__last__last_12h + fluid_out_urine__mean__last_12h + ureum__last__last_12h + creatinine__last__last_12h + arterial_blood_pressure_mean__mean__last_12h + heart_rate__mean__last_12h + hemoglobin__last__last_12h + temperature__mean__last_12h + activated_partial_thromboplastin_time__last__last_12h + bicarbonate_arterial__last__last_12h'
#I_model = 'D ~ vent_mode__hours_since_last__last_12h + pco2_arterial__mean__last_12h + po2_arterial__mean__last_12h + o2_flow__last__last_12h + o2_saturation__mean__last_12h + respiratory_rate_measured__mean__last_12h + glasgow_coma_scale_total__last__last_12h + lactate__last__last_12h + fluid_out_urine__mean__last_12h + ureum__last__last_12h + creatinine__last__last_12h + arterial_blood_pressure_mean__mean__last_12h + heart_rate__mean__last_12h + hemoglobin__last__last_12h + temperature__mean__last_12h + activated_partial_thromboplastin_time__last__last_12h + bicarbonate_arterial__last__last_12h'

I_model = (
"D ~ t0"
" + lag1_vent_mode__hours_since_last__last_12h"
" + lag1_pco2_arterial__mean__last_12h + lag1_po2_arterial__mean__last_12h"
" + lag1_o2_flow__last__last_12h + lag1_o2_saturation__mean__last_12h"
" + lag1_respiratory_rate_measured__mean__last_12h"
" + lag1_glasgow_coma_scale_total__last__last_12h"
" + lag1_lactate__last__last_12h"
" + lag1_fluid_out_urine__mean__last_12h"
" + lag1_ureum__last__last_12h + lag2_ureum__last__last_12h"
" + lag1_creatinine__last__last_12h + lag2_creatinine__last__last_12h"
" + lag1_arterial_blood_pressure_mean__mean__last_12h + lag1_heart_rate__mean__last_12h"
" + lag1_hemoglobin__last__last_12h + lag2_hemoglobin__last__last_12h"
" + lag1_temperature__mean__last_12h"
" + lag1_activated_partial_thromboplastin_time__last__last_12h + lag2_activated_partial_thromboplastin_time__last__last_12h"
" + lag1_bicarbonate_arterial__last__last_12h + lag2_bicarbonate_arterial__last__last_12h"
)


In [62]:
'''def zmodel_fit_custom(zmodel, fit_data):
    z_name, x_name = re.split('~', zmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    # get feature and target data to fit zmodel
    z = fit_data[z_name].to_numpy()
    #X = fit_data[x_name].to_numpy()
    X = fit_data[x_name]

    # Initialize the LightGBM model
    #z_model_lgbm = lgb.LGBMClassifier(random_state=33)
    z_model_lgbm = lgb.LGBMClassifier(
                    #random_state=33,
                    deterministic=True,
                    force_col_wise=True,
                    n_jobs=1,
                    seed=133,
                    feature_fraction_seed=133,
                    bagging_seed=133,
                    drop_seed=133,
                    data_random_seed=133,
                )
    
    cat_features = ['vent_mode__last__last_12h']
    #X['vent_mode__last__last_12h'] = X['vent_mode__last__last_12h'].astype('category') 
    
    # Train the model on data
    z_model_lgbm.fit(
        X,
        z,
        categorical_feature=cat_features,  # Declare categorical features
        #eval_metric='auc'
    )
    #fit_rf = RandomForestRegressor()
    #fit_rf.fit(X, y)
    
    return z_model_lgbm

def zmodel_predict_custom(zmodel, new_df, fit):
    z_name, x_name = re.split('~', zmodel.replace(' ', ''))
    x_name = re.split('\+', x_name.replace(' ', ''))
    # get feature data to predict
    #X = new_df[x_name].to_numpy()
    X = new_df[x_name]
    
    # Make predictions
    prediction = fit.predict_proba(X)[:, 1]
    prediction = pd.Series(prediction)
    #prediction = np.vectorize({'1': 1, '0': 0}.get)(prediction).astype(int)
    
    # Draw categories from probabilities for a classification model
    #if y_name in ['vent_mode__hours_since_last__last_12h', 'A', 'Y']:
    #    prediction = np.vectorize(binorm_sample)(prediction)
    
    return prediction'''

"def zmodel_fit_custom(zmodel, fit_data):\n    z_name, x_name = re.split('~', zmodel.replace(' ', ''))\n    x_name = re.split('\\+', x_name.replace(' ', ''))\n    # get feature and target data to fit zmodel\n    z = fit_data[z_name].to_numpy()\n    #X = fit_data[x_name].to_numpy()\n    X = fit_data[x_name]\n\n    # Initialize the LightGBM model\n    #z_model_lgbm = lgb.LGBMClassifier(random_state=33)\n    z_model_lgbm = lgb.LGBMClassifier(\n                    #random_state=33,\n                    deterministic=True,\n                    force_col_wise=True,\n                    n_jobs=1,\n                    seed=133,\n                    feature_fraction_seed=133,\n                    bagging_seed=133,\n                    drop_seed=133,\n                    data_random_seed=133,\n                )\n\n    cat_features = ['vent_mode__last__last_12h']\n    #X['vent_mode__last__last_12h'] = X['vent_mode__last__last_12h'].astype('category') \n\n    # Train the model on data\n    z_model

In [63]:
time_points = np.max(np.unique(mimicdata[time_name])) + 1
#time_points = 200

In [64]:
time_points

np.int64(181)

In [65]:
#'vent_mode__hours_since_last__last_12h',    
#    'pco2_arterial__mean__last_12h',
#    'po2_arterial__mean__last_12h',
#    'lactate__last__last_12h',
#    'fluid_out_urine__mean__last_12h',
#    'activated_partial_thromboplastin_time__last__last_12h'

# Knight strategy
def dynamic_knight(new_df, pool, int_var, time_name, t):
    new_df.loc[new_df[time_name] == t, int_var] = 0
    condition_knight = (

        # Airway patent
        new_df['vent_mode__last__last_12h'].isin(['unknown']) &

            # Vent status not just "recently changed" (proxy for stability)
            #(new_df['vent_mode__hours_since_last__last_12h'] >= 4.0) &
        
        # --- Respiratory ---
        # R1 FiO2 <= 0.6 
        # (new_df['fio2__mean__last_12h'] <= 0.60) &
        # Proxy: low O2 flow requirement
        (new_df['o2_flow__last__last_12h'] <= 30.0) &

        # R2 SpO2 >= 95 (%)
        (new_df['o2_saturation__mean__last_12h'] >= 95.0) &

        # R3 HCO3 >= 19 (mmol/L)  (your bicarbonate is arterial, mmol/L)
        (new_df['bicarbonate_arterial__last__last_12h'] >= 19.0) &

        # R4 RR 10–30 (bpm)
        (new_df['respiratory_rate_measured__mean__last_12h'].between(10.0, 30.0)) &

        # --- Cardiovascular ---
        # C0 Systolic BP >= 100 mmHg
        # Proxy: MAP >= 75 mmHg
        (new_df['arterial_blood_pressure_mean__mean__last_12h'] >= 75.0) &

        # C1 HR 60–100 bpm
        (new_df['heart_rate__mean__last_12h'].between(60.0, 100.0)) &

        # CNS GCS >= 14
        (new_df['glasgow_coma_scale_total__last__last_12h'] >= 14.0) &

        # T 36–37.5 °C
        (new_df['temperature__mean__last_12h'].between(36.0, 37.5)) &

        # B0 Hb >= 90 g/L
        # Your hemoglobin appears to be in g/dL (mean ~9.9). Convert g/L -> g/dL: 90 g/L = 9.0 g/dL
        (new_df['hemoglobin__last__last_12h'] >= 9.0) &

        # B3 creatinine 59–104 µmol/L
        # Your creatinine appears to be in µmol/L (median ~88). Use as-is.
        (new_df['creatinine__last__last_12h'].between(59.0, 104.0)) &

        # B4 urea (BUN) 2.5–7.8 mmol/L
        # Your variable is "ureum" and appears to be in mmol/L (median ~7.85). Use as-is.
        (new_df['ureum__last__last_12h'].between(2.5, 7.8))
    )

    new_df.loc[condition_knight, int_var] = 1


In [66]:
def dynamic_discharge_aggressive_v3(new_df, pool, int_var, time_name, t):
    # Default: discharge
    new_df.loc[new_df[time_name] == t, int_var] = 1

    # --- ICU-only RED FLAGS (EARLY PHASE) ---
    # Keep this list *minimal*; anything not here is assumed manageable outside ICU (HDU/ward).
    red_flags_early = (
        # 1) Clearly invasive controlled ventilation only
        (new_df['vent_mode__last__last_12h'] == 'invasive_controlled') |

        # 2) Severe oxygenation / support (extremes only)
        (new_df['o2_saturation__mean__last_12h'] < 85.0) |
        (new_df['o2_flow__last__last_12h'] > 85.0) |

        # 3) Profound shock / severe instability (extremes only)
        (new_df['arterial_blood_pressure_mean__mean__last_12h'] < 50.0) |
        (new_df['lactate__last__last_12h'] > 6.0) |
        (new_df['heart_rate__mean__last_12h'] > 160.0) |

        # 4) Deep coma only
        (new_df['glasgow_coma_scale_total__last__last_12h'] < 6.0) |

        # 5) Catastrophic acid–base only
        (new_df['bicarbonate_arterial__last__last_12h'] < 10.0)
    )

    # IMPORTANT: do NOT block just because vent_mode == 'unknown'
    # Only block unknown if physiology suggests ICU-level respiratory failure.
    unknown_vent_extreme = (
        (new_df['vent_mode__last__last_12h'] == 'unknown') &
        (
            (new_df['o2_saturation__mean__last_12h'] < 85.0) |
            (new_df['o2_flow__last__last_12h'] > 85.0) |
            (new_df['pco2_arterial__mean__last_12h'] > 80.0) |
            (new_df['respiratory_rate_measured__mean__last_12h'] > 45.0)
        )
    )

    red_flags_early = red_flags_early | unknown_vent_extreme

    # --- HARD STEP-DOWN DEADLINE (LATE PHASE) ---
    # After this many bins, *force discharge* unless only the MOST extreme life-support needs.
    # Tune: lowering this (e.g., 20–30 bins) usually drops mortality a lot.
    late = (new_df[time_name] == t) & (t >= 24)  # 24 bins = 12 days if bin=12h

    red_flags_late = (
        (new_df['vent_mode__last__last_12h'] == 'invasive_controlled') |
        (new_df['o2_saturation__mean__last_12h'] < 80.0) |
        (new_df['arterial_blood_pressure_mean__mean__last_12h'] < 45.0) |
        (new_df['lactate__last__last_12h'] > 8.0) |
        (new_df['glasgow_coma_scale_total__last__last_12h'] < 5.0)
    )

    keep_in_icu = (red_flags_early & ~late) | (red_flags_late & late)

    new_df.loc[keep_in_icu, int_var] = 0


In [67]:
#int_descript = ['Discharge 0th point', 'Discharge 1st point', 'Discharge 2nd point', 'Discharge 3rd point', 'Discharge 4th point', 'Discharge 5th point', 'Discharge 6th point']

#int_descript = ['Discharge 6th point', 'Discharge natural course', 'Discharge safe', 'Discharge risky', 'Discharge dangerous']
#int_descript = ['Discharge extremely dangerous']
#int_descript = ['Discharge 1st day', 'Discharge 3rd day', 'Discharge 5th day', 'Discharge 7th day', 'Discharge Knight', 'DS1']
#int_descript = ['Discharge 3rd day', 'Discharge safe', 'Discharge risky', 'Discharge dangerous']
#int_descript = ['Discharge 5th day', 'Discharge Knight']

#int_descript = ['Day 1', 'Day 3', 'Day 5', 'Day 7', 'Day 14', 'Day 21', 'Day 28', 'Day 60', 'Day 85']
#int_descript = ['Day 1', 'Day 3', 'Day 5']
int_descript = ['Day 1']

#Intervention1_A = [static, np.array([1]), [0]]
#Intervention1_A = [static, np.array([0, 1]), [0, 1]]
#Intervention3_A = [static, np.array([0, 0, 1]), [0, 1, 2]]
#Intervention4_A = [static, np.array([0, 0, 0, 1]), [0, 1, 2, 3]]
#Intervention5_A = [static, np.array([0, 0, 0, 0, 1]), [0, 1, 2, 3, 4]]
#Intervention6_A = [static, np.array([0, 0, 0, 0, 0, 1]), [0, 1, 2, 3, 4, 5]]
#Intervention7_A = [static, np.array([0, 0, 0, 0, 0, 0, 1]), [0, 1, 2, 3, 4, 5, 6]]'''

#Intervention1_A = [static, np.array([0, 0, 0, 0, 0, 0,0,0,0,0,1]), [0, 1, 2, 3, 4, 5, 6,7, 8, 9,10]]

Intervention1_A = [static, np.array([0, 1]), [0, 1]]
#Intervention2_A = [static, np.array([0, 0, 0, 0, 0, 1]), [0, 1, 2, 3, 4, 5]]
#Intervention3_A = [static, np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1]), [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]
#Intervention4_A = [static, np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]), [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]]

'''# 7 days
Intervention4_A = [
    static,
    np.array([0]*13 + [1]),
    list(range(14))
]

# 14 days
Intervention5_A = [
    static,
    np.array([0]*28 + [1]),
    list(range(29))
]

Intervention6_A = [
    static,
    np.array([0]*42 + [1]),
    list(range(43))
]

Intervention7_A = [
    static,
    np.array([0]*56 + [1]),
    list(range(57))
]

Intervention8_A = [
    static,
    np.array([0]*120 + [1]),
    list(range(121))
]

Intervention9_A = [
    static,
    np.array([0]*170 + [1]),
    list(range(171))
]'''

#int_descript = ['Dynamic intervention']

g = ParametricGformula(obs_data=mimicdata_sub, id=id_, time_name=time_name,
                       covnames=covnames, covtypes=covtypes,
                       covmodels=covmodels, basecovs=basecovs,
                       trunc_params=trunc_params,
                       int_descript=int_descript,
                       time_points=time_points,
                       
                       Intervention1_A=Intervention1_A,
                       #Intervention2_A=Intervention2_A,
                       #Intervention3_A=Intervention3_A,
                       #Intervention4_A=Intervention4_A,
                       #Intervention5_A=Intervention5_A,
                       #Intervention6_A=Intervention6_A,
                       #Intervention7_A=Intervention7_A,
                       #Intervention8_A=Intervention8_A,
                       #Intervention9_A=Intervention9_A,
                       
                       #Intervention5_A = [dynamic_knight],
                       #Intervention6_A = [dynamic_discharge_aggressive_v3],
                       
                       #Intervention3_A = [dynamic_risky],
                       #Intervention4_A = [dynamic_dangerous],

                       #Intervention1_A=[super_dynamic_dangerous],
                       
                       outcome_name=outcome_name, 
                       zmodel=zmodel,
                       #ymodel=zmodel,
                       outcome_type=outcome_type,
                       
                       zmodel_fit_custom=None, #zmodel_fit_custom, 
                       zmodel_predict_custom=None, #zmodel_predict_custom,

                       
                       I_name=I_name,
                       I_model=I_model,
                       
                       ref_int=0,
                       model_fits=True,
                       n_simul = 2000,
                       #censor_name = censor_name, censor_model = censor_model,
                       
                       #nsamples=10, 
                       parallel=True, ncores=58,
                       save_path = "results/",
                       save_results=True,
                       
                       seed = 42,
                       
                       #ipw_cutoff_quantile = 0.99,
                       )

ValueError: covnames, covtypes and covmodels are unequal lengths.

In [ ]:
%%time
g.fit()

# Inspecting time of discharge 

In [ ]:
NC = g.summary_dict['sim_data']['Natural course']

In [ ]:
NC

In [ ]:
NC.columns

# Inspecting Y across LOS

In [ ]:
# original data
disc_alive = (mimicdata.groupby("admission_id").tail(1).query("A==1 and D==0")["admission_id"])
Z_orig = mimicdata.groupby("admission_id").tail(1).set_index("admission_id").loc[disc_alive, "Z"].mean()

# simulated NC
disc_alive_sim = NC.loc[NC.A==1]["admission_id"]
Z_sim = NC.loc[NC.admission_id.isin(disc_alive_sim)]["Y_hat"].mean()

print(Z_orig, Z_sim)


In [ ]:
(
    mimicdata.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['Y']
             .mean()
)

In [ ]:
(
    mimicdata.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['D']
             .mean()
)

In [ ]:
(
    mimicdata.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['Z']
             .mean()
)

In [ ]:
# get last row per admission (with Y and t0)
last_rows = mimicdata.groupby("admission_id").tail(1)[["admission_id", "t0", "Y"]]

# compute LOS (t0 starts at 0, so add 1)
last_rows["LOS"] = last_rows["t0"] + 1

# mean Y per LOS
mean_y_per_los = last_rows.groupby("LOS")["Y"].mean()

print(mean_y_per_los)


In [ ]:
NC = g.summary_dict['sim_data']['Natural course']

# get last row per admission (with Y and t0)
last_rows = NC.groupby("admission_id").tail(1)[["admission_id", "t0", "Y_hat"]]

# compute LOS (t0 starts at 0, so add 1)
last_rows["LOS"] = last_rows["t0"] + 1

# mean Y per LOS
mean_y_per_los = last_rows.groupby("LOS")["Y_hat"].mean()

print(mean_y_per_los)

In [ ]:
NC = g.summary_dict['sim_data']['Natural course']

In [ ]:
NC.groupby('admission_id')['t0'].max().max()

In [ ]:
DS = g.summary_dict['sim_data']['Discharge Knight']

In [ ]:
# Compute LOS per admission
los_per_admission = NC.groupby('admission_id')['t0'].max() + 1  # +1 since t0 starts at 0

# Plot histogram
plt.figure(figsize=(8, 5))
los_per_admission.hist(bins=range(1, los_per_admission.max() + 2), edgecolor='black')
plt.xlabel("Length of Stay (number of time points)")
plt.ylabel("Number of Admissions")
plt.title("Distribution of Length of Stay")
plt.grid(True)
plt.tight_layout()
plt.savefig("los_NC.pdf")
plt.show()

In [ ]:
los_per_admission = DS.groupby('admission_id')['t0'].max() + 1

In [ ]:
# Compute LOS (number of time points) per admission
los_per_admission = DS.groupby('admission_id')['t0'].max() + 1  # +1 since t0 starts at 0

# Plot histogram
plt.figure(figsize=(8, 5))
los_per_admission.hist(bins=range(1, los_per_admission.max() + 2), edgecolor='black')
plt.xlabel("Length of Stay (number of time points)")
plt.ylabel("Number of Admissions")
plt.title("Distribution of Length of Stay")
plt.grid(True)
plt.tight_layout()
plt.savefig("los_mimic.pdf")
plt.show()

In [ ]:
DS = g.summary_dict['sim_data']['Discharge safe']
#DR = g.summary_dict['sim_data']['Discharge risky']
#DD = g.summary_dict['sim_data']['Discharge dangerous']

In [ ]:
DS

In [ ]:
# Compute LOS
los_ds = DS.groupby('admission_id')['t0'].max() + 1
#los_dr = DR.groupby('admission_id')['t0'].max() + 1
#los_dd = DD.groupby('admission_id')['t0'].max() + 1

# Create vertically stacked subplots
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(10, 10), sharex=True)

# Plot for DS
axs[0].hist(los_ds, bins=range(1, 200), edgecolor='black', color='skyblue')
axs[0].set_title('LOS Distribution: DS')
axs[0].set_xlabel('Length of Stay (number of time points)')
axs[0].set_ylabel('Number of Admissions')
axs[0].grid(True)

# Plot for DR
axs[1].hist(los_dr, bins=range(1, 200), edgecolor='black', color='salmon')
axs[1].set_title('LOS Distribution: DR')
axs[1].set_xlabel('Length of Stay (number of time points)')
axs[1].set_ylabel('Number of Admissions')
axs[1].grid(True)

# Plot for DD
axs[2].hist(los_dd, bins=range(1, 200), edgecolor='black', color='salmon')
axs[2].set_title('LOS Distribution: DD')
axs[2].set_xlabel('Length of Stay (number of time points)')
axs[2].set_ylabel('Number of Admissions')
axs[2].grid(True)

# Final layout
plt.tight_layout()
plt.savefig("los_comp.pdf")
plt.show()


# Inspect percentage of in-icu death

In [ ]:
(
    mimicdata.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['D']
             .mean()
)

In [ ]:
last_rows = (
    NC.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)
)
((last_rows['A'] == 0) & (last_rows['Py'] == 1)).mean()

In [ ]:
last_rows = (
    DS.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)
)
((last_rows['A'] == 0) & (last_rows['Py'] == 1)).mean()

In [ ]:
NC.A

In [ ]:
(
    NC.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['D']
             .mean()
)

# Mean Y per LOS

In [ ]:
# Step 1: Get the last row for each admission (i.e., end of follow-up)
last_rows = mimicdata.sort_values("t0").groupby("admission_id").tail(1)

# Step 2: Calculate LOS for each admission
last_rows["los"] = mimicdata.groupby("admission_id")["t0"].nunique().values

# Step 3: Group by LOS and compute mean Y
mean_y_per_los = last_rows.groupby("los")["Y"].mean().reset_index()

# Step 4: Display
print(mean_y_per_los)


In [ ]:
mean_y_per_los.Y.mean()

In [ ]:
IPW = g.summary_dict['IP_weights']

In [ ]:
NC

# Distribution of GCS when A=1

In [ ]:
DI = g.summary_dict['sim_data']['Discharge extremely dangerous']

In [ ]:
DI

In [ ]:
# Filter data where A == 1
subset = DI[DI['A'] == 1]

# Plot distribution of X
plt.figure(figsize=(8, 5))
subset['glasgow_coma_scale_total__last__last_12h'].hist(bins=30, edgecolor='black')
plt.xlabel("L values (when A=1)")
plt.ylabel("Frequency")
plt.title("Distribution of X for A=1")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Filter data where A == 1
subset = NC[NC['A'] == 1]

# Plot distribution of X
plt.figure(figsize=(8, 5))
subset['glasgow_coma_scale_total__last__last_12h'].hist(bins=30, edgecolor='black')
plt.xlabel("L values (when A=1)")
plt.ylabel("Frequency")
plt.title("Distribution of X for A=1")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
NC

In [ ]:
# Filter rows where A == 1
subset = DI[DI['A'] == 1]

# Loop through each variable in L
for var in picked_for_L:
    print(f"\n--- Variable: {var} ---")
    
    if pd.api.types.is_numeric_dtype(subset[var]):
        # For continuous variable: show min, max, mean
        print("Type: Continuous")
        print("Min:", subset[var].min())
        print("Max:", subset[var].max())
        print("Mean:", round(subset[var].mean(), 2))
    else:
        # For categorical variable: show value counts
        print("Type: Categorical")
        print(subset[var].value_counts(dropna=False))


# Discharge safe analysis

In [ ]:
DS_filtered = DS.loc[
    ~DS["admission_id"].isin(
        DS.loc[(DS["t0"] == 179) & (DS["A"] == 1), "admission_id"]
    )
].copy()


In [ ]:
(
    DS.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['Py']
             .mean()
)

In [ ]:
result = (
    DS.loc[(DS["t0"] > 170) & (DS["A"] == 1)]
      .groupby("t0")["admission_id"]
      .nunique()
      .sort_values(ascending=False)
)

result


# Sanity checks

In [ ]:
mimicdata = mimicdata.rename(columns={'Y': 'Y_hat'})

In [ ]:
NC = g.summary_dict['sim_data']['Natural course']
D3 = g.summary_dict['sim_data']['Discharge 5th day']
#DS = g.summary_dict['sim_data']['Discharge Knight']

In [ ]:
def last_rows(df, id_col="admission_id", time_col="t0"):
    """
    Return the last row per id_col, defined by max(time_col).
    If ties exist, takes the last one after sorting.
    """
    out = (df.sort_values([id_col, time_col])
             .groupby(id_col, as_index=False)
             .tail(1)
             .reset_index(drop=True))
    return out

# In-ICU deaths

In [ ]:
mimic_D = (
    mimicdata.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['D']
             .mean()
)

last_rows = (
    NC.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)
)
NC_D = ((last_rows['A'] == 0) & (last_rows['Y_hat'] == 1)).mean()

In [ ]:



last_rows = (
    D3.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)
)
D3_D = ((last_rows['A'] == 0) & (last_rows['Py'] == 1)).mean()


last_rows = (
    DS.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)
)
DS_D = ((last_rows['A'] == 0) & (last_rows['Py'] == 1)).mean()

In [ ]:
round(mimic_D, 3), round(NC_D, 3)

In [ ]:
(round(mimic_D, 3), round(NC_D, 3), round(D3_D, 3), round(DS_D, 3)) 

# Post-discharge deaths

In [ ]:
mimic_Z = (
    mimicdata.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)['Z']
             .mean()
)

last_rows = (
    NC.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)
)
NC_Z = ((last_rows['A'] == 1) & (last_rows['Y_hat'] == 1)).mean()


In [ ]:



last_rows = (
    D3.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)
)
D3_Z = ((last_rows['A'] == 1) & (last_rows['Py'] == 1)).mean()


last_rows = (
    DS.sort_values(['admission_id','t0'])
             .groupby('admission_id')
             .tail(1)
)
DS_Z = ((last_rows['A'] == 1) & (last_rows['Py'] == 1)).mean()

In [ ]:
(round(mimic_Z, 3), round(NC_Z, 3))

In [ ]:
(round(mimic_Z, 3), round(NC_Z, 3), round(D3_Z, 3), round(DS_Z, 5)) 

In [ ]:
simdata = NC.copy()

# Comparison of covariate distributions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import ceil

# -----------------------------
# Helper: last row per admission
# -----------------------------
def last_rows(df, id_col="admission_id", time_col="t0"):
    return (df.sort_values([id_col, time_col])
              .groupby(id_col)
              .tail(1)
              .reset_index(drop=True))

# -----------------------------
# Ax-level plotting helpers
# -----------------------------
def ax_numeric_overlay(ax, m_vals, s_vals, name, bins=30):
    m_vals = pd.to_numeric(pd.Series(m_vals), errors="coerce").dropna().values
    s_vals = pd.to_numeric(pd.Series(s_vals), errors="coerce").dropna().values
    if len(m_vals) == 0 or len(s_vals) == 0:
        ax.set_visible(False)
        return None

    h1 = ax.hist(m_vals, bins=bins, density=True, alpha=0.5, label="mimic-iv")
    h2 = ax.hist(s_vals, bins=bins, density=True, alpha=0.5, label="natural course")
    ax.set_title(name, fontsize=9)
    ax.tick_params(axis="both", labelsize=8)
    return ax.get_legend_handles_labels()

def ax_categorical_bars(ax, m_ser, s_ser, name, top_k=15):
    m_ser = pd.Series(m_ser).astype("object")
    s_ser = pd.Series(s_ser).astype("object")

    m_p = m_ser.value_counts(normalize=True, dropna=False)
    s_p = s_ser.value_counts(normalize=True, dropna=False)

    cats = list((m_p.add(s_p, fill_value=0))
                .sort_values(ascending=False)
                .head(top_k)
                .index)

    if len(cats) == 0:
        ax.set_visible(False)
        return None

    m_vals = [m_p.get(c, 0.0) for c in cats]
    s_vals = [s_p.get(c, 0.0) for c in cats]

    x = np.arange(len(cats))
    width = 0.4

    b1 = ax.bar(x - width/2, m_vals, width, label="mimic-iv")
    b2 = ax.bar(x + width/2, s_vals, width, label="natural course")

    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in cats], rotation=45, ha="right", fontsize=7)
    ax.set_title(name, fontsize=9)
    ax.tick_params(axis="y", labelsize=8)
    return ax.get_legend_handles_labels()

# -----------------------------
# Prepare data
# -----------------------------
m_last = last_rows(mimicdata)
s_last = last_rows(simdata)

# -----------------------------
# Define covariates
# -----------------------------
numeric_covariates = list(picked_L)

categorical_covariates = [
    "vent_mode__last__last_12h"
]

# -----------------------------
# Subgroup: in-ICU death only
# (Assumes D=1 indicates in-ICU death; keep A=0 to exclude discharged)
# -----------------------------
mask_m = (m_last["A"] == 1) & (m_last["Z"] == 1)
mask_s = (s_last["A"] == 1) & (s_last["Z_hat"] == 1)

m_sub = m_last.loc[mask_m]
s_sub = s_last.loc[mask_s]

print(f"IN-ICU death subgroup | mimic n={len(m_sub)} | sim n={len(s_sub)}")

# -----------------------------
# Build plot list in left-to-right order
# -----------------------------
plot_specs = []
for var in numeric_covariates:
    if var in m_sub.columns and var in s_sub.columns:
        plot_specs.append(("num", var))

for var in categorical_covariates:
    if var in m_sub.columns and var in s_sub.columns:
        plot_specs.append(("cat", var))

k = len(plot_specs)
if k == 0:
    raise ValueError("No covariates found in both datasets for the selected subgroup.")

# -----------------------------
# Grid layout (left-to-right)
# -----------------------------
ncols = 3
nrows = ceil(k / ncols)

fig_w = 5.2 * ncols
fig_h = 3.6 * nrows
fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), dpi=200, constrained_layout=True)
axes = np.array(axes).reshape(-1)

legend_handles = None
legend_labels = None

for i, (kind, var) in enumerate(plot_specs):
    ax = axes[i]
    if kind == "num":
        out = ax_numeric_overlay(
            ax,
            m_sub[var],
            s_sub[var],
            name=f"{var}"
        )
    else:
        out = ax_categorical_bars(
            ax,
            m_sub[var],
            s_sub[var],
            name=f"{var}"
        )

    # Capture legend handles once for a global legend
    if out is not None and legend_handles is None:
        legend_handles, legend_labels = out

# Hide unused axes
for j in range(k, len(axes)):
    axes[j].set_visible(False)

# Global title + global legend
fig.suptitle("Covariate distribution comparison at in-ICU death (last row per admission)", fontsize=12)

if legend_handles is not None:
    fig.legend(
        legend_handles, legend_labels,
        loc="upper center",
        ncol=2,
        frameon=False,
        bbox_to_anchor=(0.5, 1.02)
    )

plt.show()


# SMD plots